In [1]:
# %pip install pandas numpy matplotlib seaborn scikit-learn

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report
import warnings

warnings.filterwarnings('ignore')

In [3]:
# Load the CSV files
teams = pd.read_csv('data/mteams.csv')
sub = pd.read_csv('final_submission_2025.csv')

# Display the first few rows of each dataframe
print(teams.head())
print(sub.head())

   TeamID     TeamName  FirstD1Season  LastD1Season
0    1101  Abilene Chr           2014          2025
1    1102    Air Force           1985          2025
2    1103        Akron           1985          2025
3    1104      Alabama           1985          2025
4    1105  Alabama A&M           2000          2025
               ID      Pred
0  2025_1101_1102  0.797561
1  2025_1101_1103  0.075156
2  2025_1101_1104  0.177275
3  2025_1101_1105  0.330844
4  2025_1101_1106  0.531553


In [4]:
# Split the 'ID' column into three separate columns
sub[['Year', 'Team1', 'Team2']] = sub['ID'].str.split('_', expand=True)

# Display the updated dataframe
print(sub.head())

               ID      Pred  Year Team1 Team2
0  2025_1101_1102  0.797561  2025  1101  1102
1  2025_1101_1103  0.075156  2025  1101  1103
2  2025_1101_1104  0.177275  2025  1101  1104
3  2025_1101_1105  0.330844  2025  1101  1105
4  2025_1101_1106  0.531553  2025  1101  1106


In [5]:
# Convert Team1 and Team2 columns to integers
sub['Team1'] = sub['Team1'].astype(int)
sub['Team2'] = sub['Team2'].astype(int)

# Merge sub with teams to get Team1Name
sub = sub.merge(teams[['TeamID', 'TeamName']], left_on='Team1', right_on='TeamID', how='left').rename(columns={'TeamName': 'Team1Name'})

# Merge sub with teams to get Team2Name
sub = sub.merge(teams[['TeamID', 'TeamName']], left_on='Team2', right_on='TeamID', how='left').rename(columns={'TeamName': 'Team2Name'})

# Drop the TeamID columns as they are no longer needed
sub = sub.drop(columns=['TeamID_x', 'TeamID_y'])

# Display the updated dataframe
print(sub.head())

               ID      Pred  Year  Team1  Team2    Team1Name    Team2Name
0  2025_1101_1102  0.797561  2025   1101   1102  Abilene Chr    Air Force
1  2025_1101_1103  0.075156  2025   1101   1103  Abilene Chr        Akron
2  2025_1101_1104  0.177275  2025   1101   1104  Abilene Chr      Alabama
3  2025_1101_1105  0.330844  2025   1101   1105  Abilene Chr  Alabama A&M
4  2025_1101_1106  0.531553  2025   1101   1106  Abilene Chr   Alabama St


In [6]:
# Add a new column for the prediction if Team1 and Team2 were switched
sub['Pred_Switched'] = 1 - sub['Pred']

# Display the updated dataframe
print(sub.head())

               ID      Pred  Year  Team1  Team2    Team1Name    Team2Name  \
0  2025_1101_1102  0.797561  2025   1101   1102  Abilene Chr    Air Force   
1  2025_1101_1103  0.075156  2025   1101   1103  Abilene Chr        Akron   
2  2025_1101_1104  0.177275  2025   1101   1104  Abilene Chr      Alabama   
3  2025_1101_1105  0.330844  2025   1101   1105  Abilene Chr  Alabama A&M   
4  2025_1101_1106  0.531553  2025   1101   1106  Abilene Chr   Alabama St   

   Pred_Switched  
0       0.202439  
1       0.924844  
2       0.822725  
3       0.669156  
4       0.468447  


In [7]:
result = sub[(sub['Team1'] == 1102) & (sub['Team2'] == 1101)]['Pred'].values
if result.size > 0:
	pred_value = result[0]
	print(pred_value)
else:
	print("No matching records found.")

No matching records found.


In [8]:
team1_rows = sub[sub['Team1'] == 1277]
print(team1_rows)

                   ID      Pred  Year  Team1  Team2    Team1Name  \
46956  2025_1277_1278  0.810036  2025   1277   1278  Michigan St   
46957  2025_1277_1279  0.624171  2025   1277   1279  Michigan St   
46958  2025_1277_1280  0.551020  2025   1277   1280  Michigan St   
46959  2025_1277_1281  0.420814  2025   1277   1281  Michigan St   
46960  2025_1277_1282  0.830565  2025   1277   1282  Michigan St   
...               ...       ...   ...    ...    ...          ...   
47146  2025_1277_1476  0.850180  2025   1277   1476  Michigan St   
47147  2025_1277_1477  0.926909  2025   1277   1477  Michigan St   
47148  2025_1277_1478  0.850472  2025   1277   1478  Michigan St   
47149  2025_1277_1479  0.891867  2025   1277   1479  Michigan St   
47150  2025_1277_1480  0.878745  2025   1277   1480  Michigan St   

            Team2Name  Pred_Switched  
46956       Minnesota       0.189964  
46957     Mississippi       0.375829  
46958  Mississippi St       0.448980  
46959        Missouri      

In [9]:
michigan_team = teams[teams['TeamName'].str.contains('Michigan', case=False, na=False)]
print(michigan_team)

     TeamID     TeamName  FirstD1Season  LastD1Season
40     1141   C Michigan           1985          2025
84     1185   E Michigan           1985          2025
175    1276     Michigan           1985          2025
176    1277  Michigan St           1985          2025
343    1444   W Michigan           1985          2025


In [10]:
florida_duke_teams = teams[teams['TeamName'].isin(['Florida', 'Duke'])]
print(florida_duke_teams)

    TeamID TeamName  FirstD1Season  LastD1Season
80    1181     Duke           1985          2025
95    1196  Florida           1985          2025


In [11]:
michigan_matches = sub[(sub['Team1Name'].isin(['Michigan', 'Michigan St'])) | (sub['Team2Name'].isin(['Michigan', 'Michigan St']))]
print(michigan_matches)

                   ID      Pred  Year  Team1  Team2    Team1Name  \
166    2025_1101_1276  0.114006  2025   1101   1276  Abilene Chr   
167    2025_1101_1277  0.157966  2025   1101   1277  Abilene Chr   
528    2025_1102_1276  0.179488  2025   1102   1276    Air Force   
529    2025_1102_1277  0.156880  2025   1102   1277    Air Force   
889    2025_1103_1276  0.455357  2025   1103   1276        Akron   
...               ...       ...   ...    ...    ...          ...   
47146  2025_1277_1476  0.850180  2025   1277   1476  Michigan St   
47147  2025_1277_1477  0.926909  2025   1277   1477  Michigan St   
47148  2025_1277_1478  0.850472  2025   1277   1478  Michigan St   
47149  2025_1277_1479  0.891867  2025   1277   1479  Michigan St   
47150  2025_1277_1480  0.878745  2025   1277   1480  Michigan St   

            Team2Name  Pred_Switched  
166          Michigan       0.885994  
167       Michigan St       0.842034  
528          Michigan       0.820512  
529       Michigan St      

In [12]:
minnesota_rutgers_matches = sub[(sub['Team1Name'].isin(['Minnesota', 'Rutgers'])) | (sub['Team2Name'].isin(['Minnesota', 'Rutgers']))]
print(minnesota_rutgers_matches)

                   ID      Pred  Year  Team1  Team2    Team1Name  \
168    2025_1101_1278  0.448153  2025   1101   1278  Abilene Chr   
240    2025_1101_1353  0.365908  2025   1101   1353  Abilene Chr   
530    2025_1102_1278  0.231302  2025   1102   1278    Air Force   
602    2025_1102_1353  0.059498  2025   1102   1353    Air Force   
891    2025_1103_1278  0.790903  2025   1103   1278        Akron   
...               ...       ...   ...    ...    ...          ...   
58680  2025_1353_1476  0.500454  2025   1353   1476      Rutgers   
58681  2025_1353_1477  0.794547  2025   1353   1477      Rutgers   
58682  2025_1353_1478  0.850596  2025   1353   1478      Rutgers   
58683  2025_1353_1479  0.605741  2025   1353   1479      Rutgers   
58684  2025_1353_1480  0.883803  2025   1353   1480      Rutgers   

            Team2Name  Pred_Switched  
168         Minnesota       0.551847  
240           Rutgers       0.634092  
530         Minnesota       0.768698  
602           Rutgers      

In [13]:
# Rename columns for clarity
df = sub.rename(columns={
    'ID': 'MatchID',
    'Pred': 'PredictionScore',
    'Year': 'MatchYear',
    'Team1': 'Team1ID',
    'Team2': 'Team2ID',
    'Team1Name': 'Team1Name',
    'Team2Name': 'Team2Name',
    'Pred_Switched': 'PredictionScoreSwitched'
})

# Create a new column to indicate the winning team based on the prediction score
df['WinningTeam'] = np.where(df['PredictionScore'] > 0.50, df['Team1Name'], df['Team2Name'])

# Display the updated dataframe
print(df.head())

          MatchID  PredictionScore MatchYear  Team1ID  Team2ID    Team1Name  \
0  2025_1101_1102         0.797561      2025     1101     1102  Abilene Chr   
1  2025_1101_1103         0.075156      2025     1101     1103  Abilene Chr   
2  2025_1101_1104         0.177275      2025     1101     1104  Abilene Chr   
3  2025_1101_1105         0.330844      2025     1101     1105  Abilene Chr   
4  2025_1101_1106         0.531553      2025     1101     1106  Abilene Chr   

     Team2Name  PredictionScoreSwitched  WinningTeam  
0    Air Force                 0.202439  Abilene Chr  
1        Akron                 0.924844        Akron  
2      Alabama                 0.822725      Alabama  
3  Alabama A&M                 0.669156  Alabama A&M  
4   Alabama St                 0.468447  Abilene Chr  


In [14]:
# df = df.drop(columns=['Team1ID', 'Team2ID'])
print(df.head())

          MatchID  PredictionScore MatchYear  Team1ID  Team2ID    Team1Name  \
0  2025_1101_1102         0.797561      2025     1101     1102  Abilene Chr   
1  2025_1101_1103         0.075156      2025     1101     1103  Abilene Chr   
2  2025_1101_1104         0.177275      2025     1101     1104  Abilene Chr   
3  2025_1101_1105         0.330844      2025     1101     1105  Abilene Chr   
4  2025_1101_1106         0.531553      2025     1101     1106  Abilene Chr   

     Team2Name  PredictionScoreSwitched  WinningTeam  
0    Air Force                 0.202439  Abilene Chr  
1        Akron                 0.924844        Akron  
2      Alabama                 0.822725      Alabama  
3  Alabama A&M                 0.669156  Alabama A&M  
4   Alabama St                 0.468447  Abilene Chr  


In [15]:
# Rename columns
df = df.rename(columns={
    'PredictionScore': 'P_Score_1',
    'PredictionScoreSwitched': 'P_Score_2'
})

# Drop the MatchYear column
df = df.drop(columns=['MatchYear'])

# Display the updated dataframe
print(df.head())

          MatchID  P_Score_1  Team1ID  Team2ID    Team1Name    Team2Name  \
0  2025_1101_1102   0.797561     1101     1102  Abilene Chr    Air Force   
1  2025_1101_1103   0.075156     1101     1103  Abilene Chr        Akron   
2  2025_1101_1104   0.177275     1101     1104  Abilene Chr      Alabama   
3  2025_1101_1105   0.330844     1101     1105  Abilene Chr  Alabama A&M   
4  2025_1101_1106   0.531553     1101     1106  Abilene Chr   Alabama St   

   P_Score_2  WinningTeam  
0   0.202439  Abilene Chr  
1   0.924844        Akron  
2   0.822725      Alabama  
3   0.669156  Alabama A&M  
4   0.468447  Abilene Chr  


In [16]:
# Rearrange the columns
df = df[['WinningTeam', 'Team1Name', 'Team1ID', 'Team2Name', 'Team2ID','P_Score_1', 'P_Score_2', 'MatchID']]

# Display the updated dataframe
print(df.head())

   WinningTeam    Team1Name  Team1ID    Team2Name  Team2ID  P_Score_1  \
0  Abilene Chr  Abilene Chr     1101    Air Force     1102   0.797561   
1        Akron  Abilene Chr     1101        Akron     1103   0.075156   
2      Alabama  Abilene Chr     1101      Alabama     1104   0.177275   
3  Alabama A&M  Abilene Chr     1101  Alabama A&M     1105   0.330844   
4  Abilene Chr  Abilene Chr     1101   Alabama St     1106   0.531553   

   P_Score_2         MatchID  
0   0.202439  2025_1101_1102  
1   0.924844  2025_1101_1103  
2   0.822725  2025_1101_1104  
3   0.669156  2025_1101_1105  
4   0.468447  2025_1101_1106  


In [17]:
temple_north_texas_matches = df[((df['Team1Name'] == 'Temple') & (df['Team2Name'] == 'North Texas')) | 
                                ((df['Team1Name'] == 'North Texas') & (df['Team2Name'] == 'Temple'))]
print(temple_north_texas_matches)

      WinningTeam    Team1Name  Team1ID Team2Name  Team2ID  P_Score_1  \
53738      Temple  North Texas     1317    Temple     1396   0.483748   

       P_Score_2         MatchID  
53738   0.516252  2025_1317_1396  


In [18]:
washington_matches = df[(df['Team1Name'].str.contains('Washington', case=False, na=False)) | 
                        (df['Team2Name'].str.contains('Washington', case=False, na=False))]
print(washington_matches)

         WinningTeam      Team1Name  Team1ID       Team2Name  Team2ID  \
78      E Washington    Abilene Chr     1101    E Washington     1186   
95      G Washington    Abilene Chr     1101    G Washington     1203   
331       Washington    Abilene Chr     1101      Washington     1449   
332    Washington St    Abilene Chr     1101   Washington St     1450   
440     E Washington      Air Force     1102    E Washington     1186   
...              ...            ...      ...             ...      ...   
65626  Washington St  Washington St     1450       Stonehill     1476   
65627  Washington St  Washington St     1450  East Texas A&M     1477   
65628  Washington St  Washington St     1450        Le Moyne     1478   
65629  Washington St  Washington St     1450      Mercyhurst     1479   
65630  Washington St  Washington St     1450    West Georgia     1480   

       P_Score_1  P_Score_2         MatchID  
78      0.425736   0.574264  2025_1101_1186  
95      0.183589   0.816411  20

In [19]:
washington_as_any_team = df[(df['Team1ID'] == 1449) | (df['Team2ID'] == 1449)]
print(washington_as_any_team)


      WinningTeam    Team1Name  Team1ID       Team2Name  Team2ID  P_Score_1  \
331    Washington  Abilene Chr     1101      Washington     1449   0.487789   
693    Washington    Air Force     1102      Washington     1449   0.256253   
1054        Akron        Akron     1103      Washington     1449   0.870380   
1414      Alabama      Alabama     1104      Washington     1449   0.737756   
1773   Washington  Alabama A&M     1105      Washington     1449   0.344822   
...           ...          ...      ...             ...      ...        ...   
65596  Washington   Washington     1449       Stonehill     1476   0.508596   
65597  Washington   Washington     1449  East Texas A&M     1477   0.665995   
65598    Le Moyne   Washington     1449        Le Moyne     1478   0.424487   
65599  Washington   Washington     1449      Mercyhurst     1479   0.608448   
65600  Washington   Washington     1449    West Georgia     1480   0.804949   

       P_Score_2         MatchID  
331     0.512211

In [20]:
ohio_state_matches = df[(df['Team1Name'].str.contains('Ohio St', case=False, na=False)) | 
                        (df['Team2Name'].str.contains('Ohio St', case=False, na=False))]
print(ohio_state_matches)

      WinningTeam    Team1Name  Team1ID       Team2Name  Team2ID  P_Score_1  \
214       Ohio St  Abilene Chr     1101         Ohio St     1326   0.099546   
576       Ohio St    Air Force     1102         Ohio St     1326   0.039269   
937         Akron        Akron     1103         Ohio St     1326   0.697387   
1297      Alabama      Alabama     1104         Ohio St     1326   0.594202   
1656      Ohio St  Alabama A&M     1105         Ohio St     1326   0.067465   
...           ...          ...      ...             ...      ...        ...   
55183     Ohio St      Ohio St     1326       Stonehill     1476   0.668854   
55184     Ohio St      Ohio St     1326  East Texas A&M     1477   0.929587   
55185     Ohio St      Ohio St     1326        Le Moyne     1478   0.928530   
55186     Ohio St      Ohio St     1326      Mercyhurst     1479   0.664758   
55187     Ohio St      Ohio St     1326    West Georgia     1480   0.948043   

       P_Score_2         MatchID  
214     0.900454

In [21]:
ohio_st_not_winning = df[((df['Team1Name'] == 'Ohio St') | (df['Team2Name'] == 'Ohio St')) & (df['WinningTeam'] != 'Ohio St')]
print(ohio_st_not_winning)
teams_ohio_st_loses_to = ohio_st_not_winning['WinningTeam'].unique().tolist()
print(teams_ohio_st_loses_to)

         WinningTeam      Team1Name  Team1ID      Team2Name  Team2ID  \
937            Akron          Akron     1103        Ohio St     1326   
1297         Alabama        Alabama     1104        Ohio St     1326   
3789         Arizona        Arizona     1112        Ohio St     1326   
6232          Auburn         Auburn     1120        Ohio St     1326   
7264          Baylor         Baylor     1124        Ohio St     1326   
10639         Bryant         Bryant     1136        Ohio St     1326   
11961            BYU            BYU     1140        Ohio St     1326   
15831     Cincinnati     Cincinnati     1153        Ohio St     1326   
16462        Clemson        Clemson     1155        Ohio St     1326   
18946    Connecticut    Connecticut     1163        Ohio St     1326   
20767  CS Northridge  CS Northridge     1169        Ohio St     1326   
23722          Drake          Drake     1179        Ohio St     1326   
24012         Drexel         Drexel     1180        Ohio St     

In [22]:
df.to_csv('madness_pred.csv', index=False)